# Práctica — Clase 12. Casos de uso reales

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

En la clase viste el mini-AlphaZero para Tres en Raya: autojuego, búsqueda tipo Búsqueda en Árbol de Monte Carlo (MCTS, por sus siglas en inglés *Monte Carlo Tree Search*) y mejora iterativa de la política. En esta práctica vas más allá: primero programas tú misma o tú mismo, desde cero, el corazón de esa búsqueda; después adaptas todo el esqueleto a un juego distinto, Conecta 4 reducido.

**Dos ejercicios:**

1. Completa el paso de selección y expansión de MCTS, usando *Upper Confidence bound for Trees* (UCT)
2. Adapta el tablero y las reglas de victoria a una versión reducida de Conecta 4, y compara cuántas simulaciones hacen falta para dejar de perder contra un jugador aleatorio

**Antes de empezar, ejecuta las dos celdas siguientes para tener listo el andamiaje del árbol de búsqueda.** No hace falta que las edites.


In [ ]:
import math
import random

import numpy as np

np.random.seed(0)
random.seed(0)


In [ ]:
# --- Tres en Raya: identico al de la clase ---
LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]


def tablero_nuevo():
    return tuple([0] * 9)  # 0 = vacio, 1 = X, -1 = O


def movimientos_validos(tablero):
    return [i for i, v in enumerate(tablero) if v == 0]


def aplicar_movimiento(tablero, pos, jugador):
    nuevo = list(tablero)
    nuevo[pos] = jugador
    return tuple(nuevo)


def resultado(tablero):
    """Devuelve +1 si gana X, -1 si gana O, 0 si hay empate, None si sigue."""
    for a, b, c in LINEAS_GANADORAS:
        suma = tablero[a] + tablero[b] + tablero[c]
        if suma == 3:
            return 1
        if suma == -3:
            return -1
    if 0 not in tablero:
        return 0
    return None


def heuristica(tablero):
    """Sustituta minima de la red de valor de AlphaZero."""
    puntuacion = 0
    for linea in LINEAS_GANADORAS:
        valores = [tablero[i] for i in linea]
        piezas_x = valores.count(1)
        piezas_o = valores.count(-1)
        if piezas_o == 0:
            puntuacion += piezas_x ** 2
        if piezas_x == 0:
            puntuacion -= piezas_o ** 2
    return math.tanh(puntuacion / 6.0)


tabla_valores = {}


def valor_estimado(tablero):
    if tablero in tabla_valores:
        return tabla_valores[tablero]
    return heuristica(tablero)


# --- El nodo del arbol de busqueda: ya creado, no hace falta tocarlo ---
class Nodo:
    __slots__ = ("tablero", "jugador", "hijos", "N", "W", "expandido")

    def __init__(self, tablero, jugador):
        self.tablero = tablero
        self.jugador = jugador  # jugador (+1 o -1) que debe mover aqui
        self.hijos = {}
        self.N = 0
        self.W = 0.0
        self.expandido = False

    def valor_medio(self):
        return self.W / self.N if self.N > 0 else 0.0


def ucb_score(padre, hijo, c=1.4):
    """Formula UCT: combina Q (que tan bueno parece) con U (bono de
    exploracion que crece si el hijo se ha visitado poco)."""
    q = hijo.valor_medio() * padre.jugador
    u = c * math.sqrt(math.log(padre.N + 1) / (hijo.N + 1))
    return q + u


## Ejercicio 1 — Completa la selección y expansión de MCTS

Ya tienes el andamiaje: el `Nodo`, la fórmula `ucb_score` (UCT) y las reglas del Tres en Raya. Te falta la función más importante de todas, `simular`, que ejecuta un ciclo completo de selección, expansión, evaluación y retropropagación.

Debajo tienes `simular` con dos huecos marcados con `# TODO`. Fíjate en lo que ya está resuelto: si la partida ya terminó (`resultado` no es `None`), el nodo actualiza `N` y `W` con el resultado real y no hace falta tocar nada ahí.

**Tu tarea es completar los dos `# TODO`:**

1. **Expansión** (dentro de `if not nodo.expandido:`): crea un `Nodo` hijo para cada movimiento válido desde `nodo.tablero`, y guárdalo en `nodo.hijos`. No olvides marcar `nodo.expandido = True` al terminar
2. **Selección** (después del bloque anterior): elige, de entre `nodo.hijos`, el hijo con mejor `ucb_score`, y continúa la simulación recursivamente sobre él, guardando el resultado en la variable `v`

**Pistas**

- Para expansión: recorre `movimientos_validos(nodo.tablero)`, aplica cada movimiento con `aplicar_movimiento(nodo.tablero, m, nodo.jugador)` y crea el hijo con `Nodo(tablero_hijo, -nodo.jugador)` — fíjate en que el hijo hereda el turno del jugador contrario
- Guarda cada hijo así: `nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)`
- Para selección: `mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))` te da la clave del mejor hijo
- Sigue la simulación con una llamada recursiva: `v = simular(nodo.hijos[mejor_m])`

**Qué esperar mientras completas el ejercicio**: con los dos `# TODO` vacíos, `busqueda_mcts(tablero, jugador, num_simulaciones=1)` no falla (solo hace una expansión de la raíz), pero devuelve un diccionario de visitas vacío. En cuanto subas a más de una simulación, si la expansión sigue vacía seguirás viendo un diccionario vacío; si ya completaste la expansión pero no la selección, verás un error de Python indicándote que falta una variable — esa es la señal de que te toca completar selección.


In [ ]:
def simular(nodo):
    """Un ciclo de seleccion, expansion, evaluacion y retropropagacion."""
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        # TODO: expansion. Para cada movimiento valido en nodo.tablero, crea
        # un Nodo hijo con el tablero resultante y el jugador contrario, y
        # guardalo en nodo.hijos[m]. No olvides poner nodo.expandido = True
        # al final.

        # Evaluacion con la tabla de valores/heuristica (esto ya esta hecho)
        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    # TODO: seleccion. Elige, de entre nodo.hijos, el hijo con mejor
    # ucb_score(nodo, hijo), y continua la simulacion recursivamente sobre
    # el, guardando el resultado en la variable `v`.

    # Retropropagacion (esto ya esta hecho, una vez tengas `v`)
    nodo.N += 1
    nodo.W += v
    return v


def busqueda_mcts(tablero, jugador, num_simulaciones=60):
    """Devuelve {movimiento: visitas} tras num_simulaciones simulaciones."""
    raiz = Nodo(tablero, jugador)
    for _ in range(num_simulaciones):
        simular(raiz)
    return {m: hijo.N for m, hijo in raiz.hijos.items()}


**Ejecuta la celda siguiente para probar tu implementación.** Con el tablero vacío y 60 simulaciones, la casilla 4 (el centro) debería acumular claramente más visitas que las demás, porque participa en cuatro líneas ganadoras.


In [ ]:
visitas_prueba = busqueda_mcts(tablero_nuevo(), 1, num_simulaciones=60)
print("Visitas por movimiento:", visitas_prueba)


<details>
<summary>💡 Ver solución</summary>

```python
def simular(nodo):
    """Un ciclo de seleccion, expansion, evaluacion y retropropagacion."""
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        # Expansion: un hijo por cada movimiento valido
        for m in movimientos_validos(nodo.tablero):
            tablero_hijo = aplicar_movimiento(nodo.tablero, m, nodo.jugador)
            nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)
        nodo.expandido = True

        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    # Seleccion: bajamos por el hijo con mejor puntuacion UCT
    mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))
    v = simular(nodo.hijos[mejor_m])

    nodo.N += 1
    nodo.W += v
    return v
```
</details>


## Ejercicio 2 — Adapta el esqueleto a Conecta 4 reducido

Ahora reutilizas el mismo patrón —autojuego, MCTS, tabla de valores— en un juego distinto: una versión reducida de Conecta 4, en un tablero de 4x4, donde ganas conectando tres fichas en línea (en vez de cuatro, como en el Conecta 4 real).

**Para que puedas avanzar aunque el ejercicio uno no esté terminado**, la celda siguiente trae una copia ya completa y funcionando del motor de búsqueda (`Nodo`, `ucb_score`, `simular`, `busqueda_mcts`, `elegir_movimiento`, autojuego y evaluación), y también trae ya adaptados el tablero, los movimientos válidos (con gravedad: una ficha cae a la casilla libre más baja de su columna, como en el Conecta 4 real) y la heurística.

**Lo único que te toca completar es `resultado(tablero)`**, la función que decide si alguien ganó. Debajo tienes la firma con un `# TODO` y la comprobación de empate ya resuelta.

**Pistas**

- La lógica es la misma que en el Tres en Raya de la clase: recorre `LINEAS_GANADORAS` (ya generadas para ti, para el tablero 4x4) y suma los tres valores de cada línea
- Conectar tres fichas propias suma +3 (gana X) o -3 (gana O), exactamente igual que en el Tres en Raya, porque una línea sigue siendo de tres casillas
- Si ninguna línea da +3 ni -3, sigue comprobando: puede que el tablero esté lleno (empate, ya resuelto) o que la partida continúe (`None`, ya resuelto)

**Qué esperar mientras completas el ejercicio**: con el `# TODO` vacío, `resultado` nunca detecta una victoria, así que todas las partidas de la celda de comparación terminarán en empate tras llenarse el tablero (16 movimientos). Eso no da ningún error, pero tampoco te deja ver el efecto real de la búsqueda: es la señal de que aún te falta completar la comprobación de victoria.


In [ ]:
# --- Conecta 4 reducido: tablero 4x4, conecta tres en linea ---
FILAS, COLUMNAS = 4, 4


def indice(fila, col):
    return fila * COLUMNAS + col


def generar_lineas_ganadoras_4x4():
    """Todas las lineas de tres casillas seguidas: horizontales, verticales
    y las dos diagonales, dentro de un tablero de 4x4."""
    lineas = []
    for f in range(FILAS):
        for c in range(COLUMNAS - 2):
            lineas.append((indice(f, c), indice(f, c + 1), indice(f, c + 2)))
    for c in range(COLUMNAS):
        for f in range(FILAS - 2):
            lineas.append((indice(f, c), indice(f + 1, c), indice(f + 2, c)))
    for f in range(FILAS - 2):
        for c in range(COLUMNAS - 2):
            lineas.append((indice(f, c), indice(f + 1, c + 1), indice(f + 2, c + 2)))
    for f in range(FILAS - 2):
        for c in range(2, COLUMNAS):
            lineas.append((indice(f, c), indice(f + 1, c - 1), indice(f + 2, c - 2)))
    return lineas


LINEAS_GANADORAS = generar_lineas_ganadoras_4x4()


def tablero_nuevo():
    return tuple([0] * (FILAS * COLUMNAS))


def movimientos_validos(tablero):
    # una columna es valida si su casilla superior todavia esta vacia
    return [c for c in range(COLUMNAS) if tablero[indice(0, c)] == 0]


def aplicar_movimiento(tablero, col, jugador):
    """La ficha cae a la casilla libre mas baja de la columna elegida."""
    nuevo = list(tablero)
    for f in range(FILAS - 1, -1, -1):
        if nuevo[indice(f, col)] == 0:
            nuevo[indice(f, col)] = jugador
            break
    return tuple(nuevo)


def resultado(tablero):
    """Devuelve +1 si gana X, -1 si gana O, 0 si hay empate, None si sigue."""
    # TODO: recorre LINEAS_GANADORAS y comprueba si alguna linea suma +3
    # (gana X) o -3 (gana O). Pista: suma = tablero[a] + tablero[b] + tablero[c]

    if 0 not in tablero:
        return 0
    return None


def heuristica(tablero):
    puntuacion = 0
    for linea in LINEAS_GANADORAS:
        valores = [tablero[i] for i in linea]
        piezas_x = valores.count(1)
        piezas_o = valores.count(-1)
        if piezas_o == 0:
            puntuacion += piezas_x ** 2
        if piezas_x == 0:
            puntuacion -= piezas_o ** 2
    return math.tanh(puntuacion / 6.0)


tabla_valores = {}


def valor_estimado(tablero):
    if tablero in tabla_valores:
        return tabla_valores[tablero]
    return heuristica(tablero)


In [ ]:
# --- Motor de busqueda: copia completa y funcionando, para que puedas
# avanzar aunque el ejercicio uno no este terminado ---
class Nodo:
    __slots__ = ("tablero", "jugador", "hijos", "N", "W", "expandido")

    def __init__(self, tablero, jugador):
        self.tablero = tablero
        self.jugador = jugador
        self.hijos = {}
        self.N = 0
        self.W = 0.0
        self.expandido = False

    def valor_medio(self):
        return self.W / self.N if self.N > 0 else 0.0


def ucb_score(padre, hijo, c=1.4):
    q = hijo.valor_medio() * padre.jugador
    u = c * math.sqrt(math.log(padre.N + 1) / (hijo.N + 1))
    return q + u


def simular(nodo):
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        for m in movimientos_validos(nodo.tablero):
            tablero_hijo = aplicar_movimiento(nodo.tablero, m, nodo.jugador)
            nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)
        nodo.expandido = True
        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))
    v = simular(nodo.hijos[mejor_m])
    nodo.N += 1
    nodo.W += v
    return v


def busqueda_mcts(tablero, jugador, num_simulaciones):
    raiz = Nodo(tablero, jugador)
    for _ in range(num_simulaciones):
        simular(raiz)
    return {m: hijo.N for m, hijo in raiz.hijos.items()}


def elegir_movimiento(visitas, temperatura):
    movimientos = list(visitas.keys())
    conteos = np.array([visitas[m] for m in movimientos], dtype=float)
    if temperatura == 0:
        return movimientos[int(np.argmax(conteos))]
    probabilidades = conteos ** (1.0 / temperatura)
    probabilidades /= probabilidades.sum()
    return np.random.choice(movimientos, p=probabilidades)


def jugar_partida_autojuego(num_simulaciones, temperatura=0.8):
    tablero = tablero_nuevo()
    jugador = 1
    historial = []
    while resultado(tablero) is None:
        visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
        movimiento = elegir_movimiento(visitas, temperatura)
        historial.append(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        jugador = -jugador
    return historial, resultado(tablero)


def actualizar_tabla_valores(historial, resultado_final, alpha=0.5):
    for tablero in historial:
        actual = valor_estimado(tablero)
        tabla_valores[tablero] = actual + alpha * (resultado_final - actual)


def movimiento_aleatorio(tablero):
    return random.choice(movimientos_validos(tablero))


def jugar_vs_aleatorio(num_simulaciones):
    tablero = tablero_nuevo()
    jugador = 1
    while resultado(tablero) is None:
        if jugador == 1:
            visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
            movimiento = elegir_movimiento(visitas, temperatura=0)
        else:
            movimiento = movimiento_aleatorio(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        jugador = -jugador
    return resultado(tablero)


**Ejecuta la celda siguiente para comparar tres valores de `num_simulaciones` (10, 30 y 60).** Para cada uno, entrena seis generaciones (40 partidas de autojuego, 50 partidas de evaluación por generación, igual que en la clase) y observa en qué generación las derrotas llegan a cero. Con `resultado` sin completar, verás derrotas y victorias en cero durante todo el entrenamiento —solo empates— en las tres configuraciones: esa es la pista de que el TODO sigue pendiente.


In [ ]:
for num_sim in [10, 30, 60]:
    tabla_valores.clear()
    np.random.seed(0)
    random.seed(0)

    print(f"--- num_simulaciones = {num_sim} ---")
    for generacion in range(1, 7):
        for _ in range(40):
            historial, resultado_final = jugar_partida_autojuego(num_sim)
            actualizar_tabla_valores(historial, resultado_final)

        victorias, empates, derrotas = 0, 0, 0
        for _ in range(50):
            r = jugar_vs_aleatorio(num_sim)
            if r == 1:
                victorias += 1
            elif r == 0:
                empates += 1
            else:
                derrotas += 1

        print(f"  Generacion {generacion}: {victorias} victorias, {empates} empates, {derrotas} derrotas")
    print()


<details>
<summary>💡 Ver solución</summary>

```python
def resultado(tablero):
    """Devuelve +1 si gana X, -1 si gana O, 0 si hay empate, None si sigue."""
    for a, b, c in LINEAS_GANADORAS:
        suma = tablero[a] + tablero[b] + tablero[c]
        if suma == 3:
            return 1
        if suma == -3:
            return -1

    if 0 not in tablero:
        return 0
    return None
```

Con esta solución, en una ejecución de referencia (semilla en 0) el patrón es claro: con `num_simulaciones=10` todavía aparecen algunas derrotas hasta bien entrado el entrenamiento (por ejemplo, 5 derrotas en la generación 1 y otras 5 en la generación 5, y no llega a cero derrotas hasta la generación 6); con `num_simulaciones=30` las derrotas ya llegan a cero en la mayoría de generaciones, con alguna excepción puntual; y con `num_simulaciones=60` el agente deja de perder contra el jugador aleatorio desde la generación 1. Cuantas menos simulaciones de MCTS hace el agente en cada movimiento, peor es la política que produce esa búsqueda, y más autojuego necesita para compensarlo, exactamente igual que en el segundo experimento que se sugiere en el manual del curso.
</details>


## Para seguir practicando

**Prueba a subir `FILAS` y `COLUMNAS` a 5x5, manteniendo la conexión de tres en línea, y vuelve a ejecutar la comparación de `num_simulaciones`.** ¿El agente necesita más simulaciones o más generaciones para dejar de perder contra el jugador aleatorio? Coméntalo con tu equipo: es la misma pregunta que se hicieron los equipos de DeepMind al escalar de Tres en Raya de juguete hasta Go de verdad, solo que aquí el experimento entero corre en segundos.

Con esto terminas el curso completo de Reinforcement Learning (RL) desde cero. **Enhorabuena, y sigue explorando.**
